# Training on INCLUDE_50 Dataset

## Preprocessing Data

In [1]:
import pandas as pd
import numpy as np
import os

from tqdm.notebook import tqdm

In [2]:
train_data = pd.read_csv('Dataset/train.csv')
test_data = pd.read_csv('Dataset/test.csv')

train_data.sample(5)

,parent_label,label,video_path,include_50
228,People,59. Daughter,People/59. Daughter/MVI_4221.mp4,False
338,Clothes,41. Shirt,Clothes/41. Shirt/MVI_3849.mp4,False
2881,Seasons,61. Summer,Seasons/61. Summer/MVI_4566.mp4,False
3510,Animals,5. Cow,Animals/5. Cow/MVI_3074.mp4,True
2086,Places,26. University,Places/26. University/MVI_3466.mp4,False


In [3]:
# labels = train_data['label']
# train_data_path = train_data['vedio_path']


labels = []
train_path_I50 = []

for i in range(len(train_data)):
    if train_data['include_50'][i] == True:
        labels.append(train_data['label'][i])
        train_path_I50.append("Dataset\\" + train_data['video_path'][i])
        
labels = pd.Series(labels).unique()
labels = pd.Series(labels).to_list()

train_path_I50 = pd.Series(train_path_I50)

# train_path_I50.sample(5), labels.sample(5)
labels[:5]

['1. loud', '19. House', '83. big large', '91. Priest', '23. Court']

In [4]:
label_map = dict()

for i in range(len(labels)):
    split = labels[i].split(" ")
        
    label = split[1:]
    label = " ".join(label)
    
    label_map[label] = int(split[0].split(".")[0])

labels = list(label_map.keys())
label_map    

{'loud': 1,
 'House': 19,
 'big large': 83,
 'Priest': 91,
 'Court': 23,
 'train ticket': 16,
 'it': 44,
 'Shoes': 44,
 'Dog': 1,
 'Bank': 35,
 'Thank you': 55,
 'Election': 14,
 'Cow': 5,
 'Window': 28,
 'quiet': 2,
 'dry': 97,
 'long': 78,
 'Hello': 48,
 'Bird': 4,
 'Hat': 37,
 'Black': 54,
 'short': 79,
 'White': 55,
 'Fan': 53,
 'new': 91,
 'Store or Shop': 28,
 'Monday': 67,
 'Death': 2,
 'Cell phone': 54,
 'you (plural)': 46,
 'T-Shirt': 42,
 'Girl': 78,
 'Father': 61,
 'Red': 47,
 'hot': 87,
 'Fall': 64,
 'I': 40,
 'Time': 86,
 'Car': 11,
 'Good Morning': 51,
 'Summer': 61,
 'Paint': 40,
 'Teacher': 84,
 'Brother': 66,
 'good': 94,
 'happy': 3,
 'Boy': 77,
 'small little': 84,
 'Pen': 34,
 'Year': 78}

In [5]:
# Loading all the video frames
video_frames = []

for label in tqdm(labels):
    label_videos = os.listdir("MP_data/"+label)
    window = []
    
    for video in label_videos:
        res = np.load("MP_data/" + f"{label}/" + video)
        window.append(res)
    video_frames.append(window)

print(len(video_frames))
        

  0%|          | 0/50 [00:00<?, ?it/s]

50


In [6]:
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.utils import to_categorical

# Pad sequences to ensure they have the same length
max_length = max(len(seq) for seq in video_frames)
padded_video_frames = []

for frames in video_frames:
    padded_video_frames.append(pad_sequences(frames, maxlen=max_length, dtype='float32', padding='post', truncating='post'))

for i, frames in enumerate(padded_video_frames):
    print(f"Video {i} shape: {frames.shape}")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\site-packages\trait

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\site-packages\trait

AttributeError: _ARRAY_API not found

ImportError: numpy.core._multiarray_umath failed to import

ImportError: numpy.core.umath failed to import

TypeError: Unable to convert function return value to a Python type! The signature was
	() -> handle

In [7]:
max_frames = max(video.shape[0] for video in padded_video_frames)

# Pad the number of frames for each video
X = np.zeros((len(padded_video_frames), max_frames, 19, 1662), dtype='float32')

for i, video in enumerate(padded_video_frames):
    X[i, :video.shape[0]] = video

print(X.shape)

(50, 19, 19, 1662)


In [8]:
y = np.array([label_map[label] for label in labels])
y = to_categorical(y).astype(int)

print(y.shape)

(50, 98)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42) 

In [10]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Dense, LSTM, Bidirectional, Flatten, TimeDistributed, Reshape
from tensorflow.keras.models import Model



def create_model(input_shape, num_classes=50, time_steps=10):  # Added time_steps parameter
    # Input layer
    inputs = Input(shape=(time_steps, *input_shape))  # Adjusted input shape to include time steps
    
    # MobileNetV2 base
    base_model = MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    base_model.trainable = True  # Freeze the base model
    
    # Apply MobileNetV2 to each time step
    x = TimeDistributed(base_model)(inputs)
    
    # Reshape to 3D tensor for LSTM layers
    shape = tf.keras.backend.int_shape(x)
    x = Reshape((shape[1], -1))(x)
    
    # Bidirectional LSTM layers
    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    
    # Flatten the output
    x = Flatten()(x)
    
    # Fully connected layer
    x = Dense(128, activation='relu')(x)
    
    # Output layer
    outputs = Dense(num_classes, activation='softmax')(x)
    
    # Create the model
    model = Model(inputs=inputs, outputs=outputs)
    
    return model

# Example usage
input_shape = (1920, 1080, 3)
model = create_model(input_shape)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()

C:\Users\swast\AppData\Local\Temp\ipykernel_20868\543681724.py:11: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10, 1920, 1080, │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 10, 60, 34,     │     2,257,984 │
│ (TimeDistributed)               │ 1280)                  │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 10, 2611200)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 10, 128)        │ 1,336,967,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 10, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 10, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 10, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 10, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 50)             │         6,450 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,339,791,346 (4.99 GB)

 Trainable params: 1,339,757,234 (4.99 GB)

 Non-trainable params: 34,112 (133.25 KB)